In [ ]:
!pip install -q diffusers --upgrade

In [ ]:
!pip install -q invisible_watermark transformers accelerate safetensors gradio

In [ ]:
!git clone https://github.com/docty/image-creation.git

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(),'image-creation/utils'))

In [ ]:
import torch
from diffusers import  StableDiffusionImg2ImgPipeline, EulerDiscreteScheduler
from diffusers.utils import load_image
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"

scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
     
).to(device)

pipe.enable_attention_slicing()

In [ ]:
from prompt_loader import load_prompts

PROMPT_OPTIONS = load_prompts(os.path.join(os.getcwd(),'image-creation/utils/prompt_options.json'))

negative_prompt = "3D cartoon, illustration, CGI, overly clean, smooth surfaces, fantasy, bright daylight, humans, abstract, low detail, low resolution"

STRENGTH = 0.75
GUIDANCE_SCALE = 7.5

In [ ]:
examples = [
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/Dent_on_roof_of_UTA.jpg',
        'Dents, Bend, Inclination',
    ],
    
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/Stain_on_coating.jpg',
        'Coating, Damage, Stains',
    ],
    [ 
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/Coating_Damage_on_wellhead.jpg',
        'Corrosion',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/debris_1.jpg',
        'Debris',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/marine_growth_1.jpg',
        'Marine Growth',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/assets/crack_4.jpg',
        'Crack',
    ],
    
     
]

In [ ]:
def generate_image(init_image, prompt_choice):
    init_image = init_image.convert("RGB").resize((768, 512))
    prompt = PROMPT_OPTIONS[prompt_choice]

    result = pipe(
        prompt=prompt,
        image=init_image,
        strength=STRENGTH,
        guidance_scale=GUIDANCE_SCALE,
        negative_prompt=negative_prompt
    ).images[0]

    return result

with gr.Blocks(css="""
    .gr-block { padding: 1rem !important; }
    .gr-button { width: 100%; font-weight: bold; }
    .gr-image { border-radius: 12px; }
""") as demo:

    gr.Markdown("""
        <h1 style='text-align: center; color: #333;'> Image Generator</h1>
        <p style='text-align: center; max-width: 700px; margin: auto; color: #555;'>
            Upload an initial image and choose a corrosion style prompt to generate an image or choose an Example from below.
        </p>
    """)

    
    with gr.Row():
        prompt_choice = gr.Dropdown(
            choices=list(PROMPT_OPTIONS.keys()),
            label="Select Anomalies Type",
            value="Dents, Bend, Inclination"
        )
        generate_btn = gr.Button("Generate Image")

   
    with gr.Row():
        with gr.Column(scale=1):
            init_image = gr.Image(
                label="Upload Your Image",
                type="pil",
                height=400,
                width=400
            )
        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", height=300)

    generate_btn.click(
        fn=generate_image,
        inputs=[init_image, prompt_choice],
        outputs=output_image
    )

    gr.Examples(examples=examples, inputs=[init_image, prompt_choice])

demo.launch(share=True)
